# EXPERIMENT_14: Stratified Junction-Steered Mask2Former (Option 1: Patient 40 <-> Patient 38 Swap)
**Scientific Objective:**
Balance anatomical pose distribution between Train and Val by swapping Patient 40 (101 frames, 15 flipped) into Train, and Patient 38 (99 frames, 6 flipped) into Val.
Evaluate whether this enables the model to learn extreme grasper elevation mechanics during training, and test generalization on the frozen Test set (Patient 41).


In [ ]:
!pip install -q transformers
import os, sys, time, glob, json, cv2, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# NumPy 2.0 compatibility
for _a, _v in [('Inf', np.inf), ('NAN', np.nan), ('NaN', np.nan), ('PINF', np.inf), ('NINF', -np.inf)]:
    if not hasattr(np, _a):
        setattr(np, _a, _v)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# Auto-Detect Dataset Directories (Supports separate Kaggle mounted splits)
# ==============================================================================
def find_split_path(split_name, explicit_path=None):
    if explicit_path and os.path.exists(explicit_path):
        return os.path.abspath(explicit_path)
    target = split_name.lower()
    
    candidates = [
        f"/kaggle/input/datasets/khoatrytopublish/l3d-{target}/{split_name}",
        f"/kaggle/input/datasets/khoatrytopublish/l3d_{target}/{split_name}",
        f"/kaggle/input/datasets/khoatrytopublish/l3d{target}/{split_name}",
        f"/kaggle/input/l3d-{target}/{split_name}",
        f"/kaggle/input/l3d_{target}/{split_name}",
        f"/kaggle/input/l3d - {split_name}/{split_name}",
        f"/kaggle/input/l3d - {target}/{split_name}",
        f"/kaggle/input/l3d-{target}",
        f"/kaggle/input/l3d/{split_name}",
        f"/kaggle/input/laparoscopic-liver-landmark-dataset/L3D/{split_name}",
        f"/kaggle/input/l3d-dataset/L3D/{split_name}",
        f"/data/khoalq/data/L3D/{split_name}",
        f"data/L3D/{split_name}",
        f"../data/L3D/{split_name}",
        f"../../data/L3D/{split_name}",
    ]
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, "labels")):
            return os.path.abspath(c)
            
    # Recursive search under /kaggle/input
    if os.path.exists("/kaggle/input"):
        for m in glob.glob(f"/kaggle/input/**/{split_name}", recursive=True):
            if os.path.isdir(m) and os.path.exists(os.path.join(m, "labels")):
                return os.path.abspath(m)
        for m in glob.glob(f"/kaggle/input/**/{target}", recursive=True):
            if os.path.isdir(m) and os.path.exists(os.path.join(m, "labels")):
                return os.path.abspath(m)
        for p in glob.glob("/kaggle/input/**/labels", recursive=True):
            parent = os.path.dirname(p)
            if target in parent.lower():
                return os.path.abspath(parent)
                
    raise RuntimeError(f"Could not locate {split_name} split directory! Check your dataset inputs.")

TRAIN_DIR = find_split_path("Train")
VAL_DIR = find_split_path("Val")
TEST_DIR = find_split_path("Test")
if os.path.exists("/kaggle"):
    OUT_DIR = "/kaggle/working/results_exp14"
else:
    OUT_DIR = os.path.abspath("experiments/EXPERIMENT_14/results")
os.makedirs(OUT_DIR, exist_ok=True)

print("📁 Resolved Dataset Split Directories:")
print(f"   Train : {TRAIN_DIR}")
print(f"   Val   : {VAL_DIR}")
print(f"   Test  : {TEST_DIR}")
print(f"   Output: {OUT_DIR}")


In [ ]:
# ==============================================================================
# 1. Deterministic Anatomical Junction Extractor
# ==============================================================================
"""
Deterministic Anatomical Junction Extractor for L3D Dataset.
Extracts the 4 biological landmark junctions from JSON polyline annotations:
  1. J_top (idx 0): Falciform-Silhouette Root (Superior liver boundary)
  2. J_bottom (idx 1): Umbilical Notch (Inferior Falciform meets Ridge)
  3. J_lat_right (idx 2): Right Lateral Tip (Right Ridge meets Right Silhouette)
  4. J_lat_left (idx 3): Left Lateral Tip (Left Ridge meets Left Silhouette)
"""
import numpy as np

JUNCTION_NAMES = ['J_top', 'J_bottom', 'J_lat_right', 'J_lat_left']

def extract_gt_junctions(data, orig_w, orig_h, canvas_size=1024, threshold_px=100.0):
    """
    Extracts normalized coordinates (x, y) in [0, 1]^2 and binary visibility flags (4,)
    for the 4 anatomical junction keypoints from an L3D JSON annotation dict.

    Args:
        data (dict): JSON contents with 'shapes' list.
        orig_w (int): Original image width (handles 4K Patient 32).
        orig_h (int): Original image height.
        canvas_size (int): Working resolution (default 1024).
        threshold_px (float): Distance threshold in canvas pixels to consider curves connected.

    Returns:
        coords_norm (np.ndarray): Shape (4, 2), float32, normalized to [0, 1]^2.
        visibility (np.ndarray): Shape (4,), float32, in {0.0, 1.0}.
        coords_px (np.ndarray): Shape (4, 2), float32, in [0, canvas_size] pixels.
    """
    sx = float(canvas_size) / float(orig_w)
    sy = float(canvas_size) / float(orig_h)

    r_curves, s_curves, f_curves = [], [], []
    for shape in data.get('shapes', []):
        lbl = str(shape.get('label', '')).lower().strip()
        pts = np.array(shape.get('points', []), dtype=np.float32)
        if len(pts) < 2:
            continue
        # Scale to canvas coordinates
        scaled = np.column_stack([pts[:, 0] * sx, pts[:, 1] * sy])
        if lbl.startswith('r') or 'ridge' in lbl or 'rigde' in lbl:
            r_curves.append(scaled)
        elif lbl.startswith('s') or 'sil' in lbl:
            s_curves.append(scaled)
        elif lbl.startswith('f') or 'falc' in lbl or 'lig' in lbl:
            f_curves.append(scaled)

    coords_px = np.zeros((4, 2), dtype=np.float32)
    visibility = np.zeros(4, dtype=np.float32)

    # 1. J_top (Falciform meets Silhouette at superior root)
    best_d = 1e9
    j_top = None
    for fc in f_curves:
        for sc in s_curves:
            for f_pt in [fc[0], fc[-1]]:
                dists = np.linalg.norm(sc - f_pt, axis=1)
                min_idx = np.argmin(dists)
                if dists[min_idx] < best_d:
                    best_d = dists[min_idx]
                    j_top = (f_pt + sc[min_idx]) / 2.0
    if best_d <= threshold_px and j_top is not None:
        coords_px[0] = j_top
        visibility[0] = 1.0

    # 2. J_bottom (Falciform meets Ridge at Umbilical Notch)
    best_d = 1e9
    j_bot = None
    for fc in f_curves:
        for rc in r_curves:
            for f_pt in [fc[0], fc[-1]]:
                dists = np.linalg.norm(rc - f_pt, axis=1)
                min_idx = np.argmin(dists)
                if dists[min_idx] < best_d:
                    best_d = dists[min_idx]
                    j_bot = (f_pt + rc[min_idx]) / 2.0
    if best_d <= threshold_px and j_bot is not None:
        coords_px[1] = j_bot
        visibility[1] = 1.0

    # 3. J_lat_right & J_lat_left (Ridge meets Silhouette at both extremities)
    best_dr, pt_r = 1e9, None
    best_dl, pt_l = 1e9, None

    for rc in r_curves:
        for sc in s_curves:
            # Check ridge endpoints to silhouette curve
            for r_pt in [rc[0], rc[-1]]:
                d_s = np.linalg.norm(sc - r_pt, axis=1)
                idx = np.argmin(d_s)
                d = d_s[idx]
                pt = (r_pt + sc[idx]) / 2.0
                if pt[0] >= canvas_size * 0.45 and d < best_dr:
                    best_dr, pt_r = d, pt
                elif pt[0] < canvas_size * 0.45 and d < best_dl:
                    best_dl, pt_l = d, pt
            # Check silhouette endpoints to ridge curve
            for s_pt in [sc[0], sc[-1]]:
                d_r = np.linalg.norm(rc - s_pt, axis=1)
                idx = np.argmin(d_r)
                d = d_r[idx]
                pt = (s_pt + rc[idx]) / 2.0
                if pt[0] >= canvas_size * 0.45 and d < best_dr:
                    best_dr, pt_r = d, pt
                elif pt[0] < canvas_size * 0.45 and d < best_dl:
                    best_dl, pt_l = d, pt

    if best_dr <= threshold_px and pt_r is not None:
        coords_px[2] = pt_r
        visibility[2] = 1.0

    if best_dl <= threshold_px and pt_l is not None:
        coords_px[3] = pt_l
        visibility[3] = 1.0

    # Normalize coordinates to [0, 1]^2
    coords_norm = coords_px / float(canvas_size)

    return coords_norm, visibility, coords_px

In [ ]:
# ==============================================================================
# 2. Evaluation Metrics (NumPy 2.0 & Pure OpenCV)
# ==============================================================================
"""
Evaluation metrics for EXPERIMENT_5:
  - Macro Dice, IoU, ASSD per class (Ridge, Silhouette, Falciform)
  - Patient 40 diagnostic metrics
  - 4-Junction Mean Absolute Pixel Error (MAPE) & Visibility Accuracy
  - Pure OpenCV & NumPy implementation (zero external C-extension dependencies)
"""
import cv2
import numpy as np
import torch

# NumPy 2.0 compatibility
for _a, _v in [('Inf', np.inf), ('NAN', np.nan), ('NaN', np.nan), ('PINF', np.inf), ('NINF', -np.inf)]:
    if not hasattr(np, _a):
        setattr(np, _a, _v)

def compute_dice(pred_bin, target_bin, eps=1e-6):
    pred_b = (pred_bin > 0).astype(bool)
    target_b = (target_bin > 0).astype(bool)
    intersection = np.logical_and(pred_b, target_b).sum()
    total = pred_b.sum() + target_b.sum()
    if total == 0:
        return 1.0
    return float(2.0 * intersection / (total + eps))

def compute_iou(pred_bin, target_bin, eps=1e-6):
    pred_b = (pred_bin > 0).astype(bool)
    target_b = (target_bin > 0).astype(bool)
    intersection = np.logical_and(pred_b, target_b).sum()
    union = np.logical_or(pred_b, target_b).sum()
    if union == 0:
        return 1.0
    return float(intersection / (union + eps))

def compute_assd_fast(pred_bin, target_bin, max_penalty=80.0):
    """
    Computes Average Symmetric Surface Distance (ASSD) using OpenCV distanceTransform.
    Guaranteed zero external dependency issues.
    """
    p_b = (pred_bin > 0).astype(np.uint8)
    t_b = (target_bin > 0).astype(np.uint8)
    
    if p_b.sum() == 0 and t_b.sum() == 0:
        return 0.0
    if p_b.sum() == 0 or t_b.sum() == 0:
        return max_penalty
        
    try:
        # Extract 1-pixel boundary edges
        contours_p, _ = cv2.findContours(p_b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        contours_t, _ = cv2.findContours(t_b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        
        edge_p = np.zeros_like(p_b)
        edge_t = np.zeros_like(t_b)
        cv2.drawContours(edge_p, contours_p, -1, 1, 1)
        cv2.drawContours(edge_t, contours_t, -1, 1, 1)
        
        if edge_p.sum() == 0 or edge_t.sum() == 0:
            return max_penalty
            
        dist_t = cv2.distanceTransform((1 - edge_t).astype(np.uint8), cv2.DIST_L2, 3)
        dist_p = cv2.distanceTransform((1 - edge_p).astype(np.uint8), cv2.DIST_L2, 3)
        
        d_p2t = np.mean(dist_t[edge_p > 0])
        d_t2p = np.mean(dist_p[edge_t > 0])
        
        return float((d_p2t + d_t2p) / 2.0)
    except Exception:
        return max_penalty

def evaluate_frame_metrics(pred_map, target_map, pred_j_coords=None, gt_j_coords=None, gt_j_vis=None, canvas_size=1024):
    """
    Computes all semantic and junction metrics for a single 1024x1024 frame.
    """
    c_dices = []
    c_ious = []
    c_assds = []
    
    # Classes: 1=Ridge, 2=Sil, 3=Falc
    for c in [1, 2, 3]:
        p_c = (pred_map == c)
        t_c = (target_map == c)
        c_dices.append(compute_dice(p_c, t_c))
        c_ious.append(compute_iou(p_c, t_c))
        c_assds.append(compute_assd_fast(p_c, t_c))
        
    macro_dice = float(np.mean(c_dices))
    macro_iou = float(np.mean(c_ious))
    macro_assd = float(np.mean(c_assds))
    fg_dice = compute_dice(pred_map > 0, target_map > 0)
    
    res = {
        'macro_dice': macro_dice,
        'macro_iou': macro_iou,
        'macro_assd': macro_assd,
        'ridge_dice': c_dices[0],
        'sil_dice': c_dices[1],
        'falc_dice': c_dices[2],
        'fg_dice': fg_dice,
        'ridge_assd': c_assds[0],
        'sil_assd': c_assds[1],
        'falc_assd': c_assds[2]
    }
    
    # Junction metrics (if available)
    if pred_j_coords is not None and gt_j_coords is not None and gt_j_vis is not None:
        j_errs = []
        names = ['j_top_err_px', 'j_bot_err_px', 'j_lat_r_err_px', 'j_lat_l_err_px']
        for k in range(4):
            if gt_j_vis[k] > 0.5:
                err_px = float(np.linalg.norm((pred_j_coords[k] - gt_j_coords[k]) * float(canvas_size)))
                j_errs.append(err_px)
                res[names[k]] = err_px
            else:
                res[names[k]] = None
        res['mean_j_err_px'] = float(np.mean(j_errs)) if len(j_errs) > 0 else None
        
    return res

In [ ]:
# ==============================================================================
# 3. Stratified L3D Dataset Reader (Option 1 Re-stratification)
# ==============================================================================
"""
Stratified L3D PyTorch Dataset for EXPERIMENT_14 (Option 1 Re-stratified Junction-Steered Mask2Former).
Patient-level Re-stratification (Option 1: Patient_40 <-> Patient_38 Swap):
  - Train: Contains Patient_40 (101 frames) + all other non-test patients except Patient_38.
           Total: 923 frames, 83 flipped (8.99% vs population 8.92%).
  - Val:   Contains Patient_38 (99 frames), Patient_32 (15 frames), Patient_18 (6 frames).
           Total: 120 frames, 10 flipped (8.33% vs population 8.92%).
  - Test:  Official frozen test benchmark (Patients 21, 31, 41, 51). Total: 109 frames.

Features:
  - Preserves 100% architectural and metric compatibility with EXPERIMENT_5.
  - Zero disk mutation: reads from original Train and Val directories on disk without copying files.
  - Automatic environment resolution (local macOS vs. Kaggle CUDA).
"""
import os
import glob
import json
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from pathlib import Path


# Standard ImageNet statistics
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def find_split_path(split_name, explicit_path=None, base_dir=None):
    """
    Universally resolves directory paths for Train, Val, and Test splits.
    Supports explicit paths, unified root directories, and multi-dataset Kaggle mounts.
    """
    if explicit_path and os.path.exists(explicit_path):
        return os.path.abspath(explicit_path)
    target = split_name.lower()
    
    # 1. Inside base_dir if given
    if base_dir:
        for candidate in [os.path.join(base_dir, split_name), os.path.join(base_dir, target)]:
            if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'labels')):
                return os.path.abspath(candidate)
        if os.path.exists(os.path.join(base_dir, 'labels')):
            return os.path.abspath(base_dir)

    # 2. Standard known paths (including Kaggle mounted datasets)
    candidates = [
        f"/kaggle/input/datasets/khoatrytopublish/l3d-{target}/{split_name}",
        f"/kaggle/input/datasets/khoatrytopublish/l3d_{target}/{split_name}",
        f"/kaggle/input/datasets/khoatrytopublish/l3d{target}/{split_name}",
        f"/kaggle/input/l3d-{target}/{split_name}",
        f"/kaggle/input/l3d_{target}/{split_name}",
        f"/kaggle/input/l3d - {split_name}/{split_name}",
        f"/kaggle/input/l3d - {target}/{split_name}",
        f"/kaggle/input/l3d-{target}",
        f"/kaggle/input/l3d/{split_name}",
        f"/kaggle/input/laparoscopic-liver-landmark-dataset/L3D/{split_name}",
        f"/kaggle/input/l3d-dataset/L3D/{split_name}",
        f"/data/khoalq/data/L3D/{split_name}",
        f"data/L3D/{split_name}",
        f"../data/L3D/{split_name}",
        f"../../data/L3D/{split_name}",
    ]
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'labels')):
            return os.path.abspath(c)
            
    # 3. Recursive search under /kaggle/input
    if os.path.exists("/kaggle/input"):
        for m in glob.glob(f"/kaggle/input/**/{split_name}", recursive=True):
            if os.path.isdir(m) and os.path.exists(os.path.join(m, 'labels')):
                return os.path.abspath(m)
        for m in glob.glob(f"/kaggle/input/**/{target}", recursive=True):
            if os.path.isdir(m) and os.path.exists(os.path.join(m, 'labels')):
                return os.path.abspath(m)
        for p in glob.glob("/kaggle/input/**/labels", recursive=True):
            parent = os.path.dirname(p)
            if target in parent.lower():
                return os.path.abspath(parent)
                
    raise RuntimeError(f"Could not locate {split_name} split directory! Check your dataset inputs.")

def resolve_l3d_root(candidate_root=None):
    """
    Auto-detects the L3D dataset root path across local macOS, Linux, and Kaggle.
    """
    candidates = [
        candidate_root,
        "data/L3D",
        "../data/L3D",
        "../../data/L3D",
        "/kaggle/input/laparoscopic-liver-landmark-dataset/L3D",
        "/kaggle/input/l3d-dataset/L3D",
        "/kaggle/input/l3d/L3D",
        "/data/khoalq/data/L3D"
    ]
    for c in candidates:
        if c and os.path.exists(c):
            return os.path.abspath(c)
    return os.path.abspath("data/L3D")

class StratifiedL3DDataset(Dataset):
    """
    Stratified L3D Dataset for EXPERIMENT_14.
    Rebalances Train and Val anatomical pose distribution by swapping Patient_40 and Patient_38.
    """
    VAL_PATIENTS = {'patient_38', 'patient_32', 'patient_18'}
    
    def __init__(self, split='Train', data_dir=None, train_dir=None, val_dir=None, test_dir=None, image_size=1024, stroke_width=35):
        super().__init__()
        self.split = split
        self.image_size = image_size
        self.stroke_width = stroke_width
        
        if split == 'Test':
            self.test_dir = find_split_path('Test', test_dir, data_dir)
            self.items = []
            for jf in sorted(glob.glob(os.path.join(self.test_dir, 'labels', '*.json'))):
                stem = Path(jf).stem
                img_path = os.path.join(self.test_dir, 'images', f"{stem}.jpg")
                if not os.path.exists(img_path):
                    img_path = os.path.join(self.test_dir, 'images', f"{stem}.png")
                self.items.append((jf, img_path))
        else:
            self.train_dir = find_split_path('Train', train_dir, data_dir)
            self.val_dir = find_split_path('Val', val_dir, data_dir)
            
            # Pool all Train and Val files from disk without altering original filesystem
            all_files = []
            for sp_dir in [self.train_dir, self.val_dir]:
                for jf in glob.glob(os.path.join(sp_dir, 'labels', '*.json')):
                    stem = Path(jf).stem
                    img_path = os.path.join(sp_dir, 'images', f"{stem}.jpg")
                    if not os.path.exists(img_path):
                        img_path = os.path.join(sp_dir, 'images', f"{stem}.png")
                    all_files.append((jf, img_path))
            
            self.items = []
            for jf, img_path in sorted(all_files):
                stem = Path(jf).stem.lower()
                is_val = any(vp in stem for vp in self.VAL_PATIENTS)
                if split == 'Val' and is_val:
                    self.items.append((jf, img_path))
                elif split == 'Train' and not is_val:
                    self.items.append((jf, img_path))
                    
        if len(self.items) == 0:
            raise RuntimeError(f"[{split}] No annotation files found in resolved directories!")
            
        print(f"[{split} - Stratified Option 1] Loaded {len(self.items)} frames.")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        json_path, img_path = self.items[idx]
        stem = Path(json_path).stem
            
        # 1. Load image
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            raise FileNotFoundError(f"Failed to read image at: {img_path}")
            
        orig_h, orig_w = img_bgr.shape[:2]
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Resize image to canvas_size
        if orig_w != self.image_size or orig_h != self.image_size:
            img_resized = cv2.resize(img_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        else:
            img_resized = img_rgb
            
        # Normalize pixel values
        norm_img = (img_resized.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        pixel_values = torch.from_numpy(norm_img).permute(2, 0, 1).float() # (3, H, W)
        
        # 2. Parse JSON annotations
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        # Dynamic canvas size: guarantees Patient 32 4K (2160x3840) is never truncated
        data_w = data.get('imageWidth', orig_w)
        data_h = data.get('imageHeight', orig_h)
        canvas_raw = np.zeros((data_h, data_w), dtype=np.uint8)
        
        # 3. Create dense raster mask
        # Draw contours with thickness 35 on raw canvas (exact TopoNet / EXP_1 paper standard)
        shapes = data.get('shapes', [])
        for shape in shapes:
            lbl = str(shape.get('label', '')).lower().strip()
            pts = shape.get('points', [])
            if len(pts) < 2:
                continue
                
            if lbl.startswith('r') or 'ridge' in lbl or 'rigde' in lbl or 'anterior' in lbl:
                color = 1
            elif lbl.startswith('s') or 'sil' in lbl or 'margin' in lbl or 'silhouette' in lbl:
                color = 2
            elif lbl.startswith('f') or lbl.startswith('l') or 'falc' in lbl or 'lig' in lbl:
                color = 3
            else:
                color = 0
                
            if color > 0:
                for i in range(1, len(pts)):
                    pt1 = tuple(map(int, pts[i - 1]))
                    pt2 = tuple(map(int, pts[i]))
                    cv2.line(canvas_raw, pt1, pt2, color, self.stroke_width)

        # Downsample to network resolution (1024x1024) using INTER_NEAREST (exact TopoNet / EXP_1 standard)
        if data_w != self.image_size or data_h != self.image_size:
            mask = cv2.resize(canvas_raw, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        else:
            mask = canvas_raw

        mask_tensor = torch.from_numpy(mask).long() # (H, W)
        
        # 4. Extract 4 GT Anatomical Junctions
        coords_norm, vis, _ = extract_gt_junctions(data, data_w, data_h, canvas_size=self.image_size)
        junction_coords = torch.from_numpy(coords_norm).float() # (4, 2)
        junction_vis = torch.from_numpy(vis).float()           # (4,)
        
        is_p40 = ('patient_40' in stem.lower() or '_40_' in stem.lower())
        is_p38 = ('patient_38' in stem.lower() or '_38_' in stem.lower())
        
        return {
            'pixel_values': pixel_values,
            'mask': mask_tensor,
            'junction_coords': junction_coords,
            'junction_vis': junction_vis,
            'filename': f"{stem}.jpg",
            'is_patient_40': is_p40,
            'is_patient_38': is_p38
        }

# Alias for drop-in compatibility
L3DDataset = StratifiedL3DDataset

In [ ]:
# ==============================================================================
# 4. Junction Anchor Head Architecture
# ==============================================================================
"""
Junction Anchor Head for EXPERIMENT_5.
Extracts 4 biological landmark vectors and predicts (x, y) coordinates + visibility flags:
  0: J_top (Falc-Sil Root)
  1: J_bottom (Umbilical Notch)
  2: J_lat_right (Right Lateral Tip)
  3: J_lat_left (Left Lateral Tip)
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class JunctionDecoderLayer(nn.Module):
    def __init__(self, embed_dim=256, num_heads=8, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout)
        )
        
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, context):
        """
        queries: (B, 4, C)
        context: (B, S, C)
        """
        # 1. Self-attention among the 4 junction nodes (anatomical graph reasoning)
        q_norm = self.norm1(queries)
        q_self, _ = self.self_attn(q_norm, q_norm, q_norm)
        queries = queries + self.dropout(q_self)
        
        # 2. Cross-attention from context feature map
        q_norm = self.norm2(queries)
        q_cross, _ = self.cross_attn(query=q_norm, key=context, value=context)
        queries = queries + self.dropout(q_cross)
        
        # 3. Feedforward
        queries = queries + self.ffn(self.norm3(queries))
        return queries

class JunctionAnchorHead(nn.Module):
    def __init__(self, embed_dim=256, num_layers=2, num_heads=8):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_junctions = 4
        
        # 4 Learnable Anatomical Query Embeddings
        self.junction_queries = nn.Parameter(torch.randn(1, 4, embed_dim) * 0.02)
        
        # 2-Layer Transformer Decoder
        self.layers = nn.ModuleList([
            JunctionDecoderLayer(embed_dim=embed_dim, num_heads=num_heads)
            for _ in range(num_layers)
        ])
        
        # Coordinate Regression Head: (B, 4, 2) in [0, 1]^2
        self.coord_head = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )
        
        # Visibility Prediction Head: (B, 4) binary logit
        self.vis_head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        
        # Anatomical Prior Biases (logits for initial coordinate sigmoid)
        # [0]: Top (0.50, 0.20), [1]: Bot (0.50, 0.70), [2]: Lat_R (0.85, 0.40), [3]: Lat_L (0.15, 0.60)
        priors = torch.tensor([
            [0.50, 0.20],
            [0.50, 0.70],
            [0.85, 0.40],
            [0.15, 0.60]
        ], dtype=torch.float32)
        # logit(p) = log(p / (1 - p))
        prior_logits = torch.log(priors / (1.0 - priors + 1e-6))
        self.register_buffer('prior_logits', prior_logits)

    def forward(self, feature_map):
        """
        Args:
            feature_map (Tensor): (B, 256, H, W) from Pixel Decoder (e.g. stride 16).
        Returns:
            coords (Tensor): (B, 4, 2) normalized to [0, 1]^2
            vis_logits (Tensor): (B, 4) unnormalized visibility logits
            features (Tensor): (B, 4, 256) junction feature tokens
        """
        B, C, H, W = feature_map.shape
        context = feature_map.flatten(2).transpose(1, 2) # (B, H*W, C)
        
        queries = self.junction_queries.expand(B, -1, -1) # (B, 4, C)
        
        for layer in self.layers:
            queries = layer(queries, context)
            
        # Predict coordinates with anatomical prior bias
        coord_deltas = self.coord_head(queries) # (B, 4, 2)
        coords = torch.sigmoid(coord_deltas + self.prior_logits.unsqueeze(0))
        
        # Predict visibility logits
        vis_logits = self.vis_head(queries).squeeze(-1) # (B, 4)
        
        return coords, vis_logits, queries

In [ ]:
# ==============================================================================
# 5. Junction-Steered Mask2Former Architecture
# ==============================================================================
"""
Junction-Steered Mask2Former Model for EXPERIMENT_5.
Combines:
  1. Pretrained Swin-Tiny Backbone + MSDeformAttn Pixel Decoder (from HF Mask2Former)
  2. 4-Query Junction Anchor Head (predicting Top, Bottom, Lat_R, Lat_L)
  3. Dynamic Junction Cross-Attention Query Steering Block
  4. 9-Layer Mask2Former Transformer Decoder
"""
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure workspace root is on sys.path


try:
    from transformers import Mask2FormerForUniversalSegmentation, AutoImageProcessor
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

class JunctionQuerySteering(nn.Module):
    """
    Dynamic Query Steering Block:
    Allows 100 Mask2Former queries to cross-attend to the 4 junction anchor vectors.
    """
    def __init__(self, embed_dim=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.proj_q = nn.Linear(embed_dim, embed_dim)
        self.proj_k = nn.Linear(embed_dim, embed_dim)
        self.proj_v = nn.Linear(embed_dim, embed_dim)
        
        # Learnable gating scalar initialized to 0.1 for smooth gradient highway
        self.gate = nn.Parameter(torch.tensor(0.1, dtype=torch.float32))
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries, junction_features, junction_coords=None):
        """
        Args:
            queries (Tensor): (B, 100, C) base query features
            junction_features (Tensor): (B, 4, C) junction anchor tokens
            junction_coords (Tensor, optional): (B, 4, 2) normalized coordinates
        Returns:
            steered_queries (Tensor): (B, 100, C)
            attn_weights (Tensor): (B, 100, 4)
        """
        # Linear projections
        q = self.proj_q(queries)
        k = self.proj_k(junction_features)
        v = self.proj_v(junction_features)
        
        # Cross-attention: queries attend to 4 junctions
        delta_q, attn_weights = self.cross_attn(query=q, key=k, value=v)
        
        # Gated residual update: Q_steered = Norm(Q + gate * delta_Q)
        steered_queries = self.norm(queries + self.gate * self.dropout(delta_q))
        
        return steered_queries, attn_weights

class JunctionSteeredMask2Former(nn.Module):
    """
    Integrated Junction-Steered Mask2Former.
    """
    def __init__(self, model_name="facebook/mask2former-swin-tiny-ade-semantic", num_labels=4):
        super().__init__()
        if not HAS_TRANSFORMERS:
            raise ImportError("HuggingFace 'transformers' library is required. Please install it or activate your PyTorch CUDA env.")
            
        print(f"📦 Building JunctionSteeredMask2Former with base: '{model_name}'...")
        is_offline = (os.environ.get("HF_HUB_OFFLINE", "0") == "1")
        try:
            self.m2f = Mask2FormerForUniversalSegmentation.from_pretrained(
                model_name,
                num_labels=num_labels,
                ignore_mismatched_sizes=True,
                local_files_only=is_offline
            )
        except Exception:
            self.m2f = Mask2FormerForUniversalSegmentation.from_pretrained(
                model_name,
                num_labels=num_labels,
                ignore_mismatched_sizes=True,
                local_files_only=True
            )
        
        embed_dim = 256
        
        # 1. Junction Anchor Head (4 biological queries)
        self.junction_head = JunctionAnchorHead(embed_dim=embed_dim, num_layers=2, num_heads=8)
        
        # 2. Query Steering Block
        self.query_steering = JunctionQuerySteering(embed_dim=embed_dim, num_heads=8)
        
    def forward_transformer_decoder(self, multi_scale_features, mask_features, steered_query_feat):
        """
        Runs the Mask2Former Transformer Decoder directly with sample-specific steered queries,
        bypassing static queries_features.weight repeating.
        """
        tm = self.m2f.model.transformer_module
        multi_stage_features = []
        multi_stage_positional_embeddings = []
        size_list = []

        for i in range(tm.num_feature_levels):
            size_list.append(multi_scale_features[i].shape[-2:])
            multi_stage_positional_embeddings.append(
                tm.position_embedder(
                    multi_scale_features[i].shape, multi_scale_features[i].device, multi_scale_features[i].dtype, None
                ).flatten(2)
            )
            multi_stage_features.append(
                tm.input_projections[i](multi_scale_features[i]).flatten(2)
                + tm.level_embed.weight[i][None, :, None]
            )
            # Permute NxCxHW to HWxNxC
            multi_stage_positional_embeddings[-1] = multi_stage_positional_embeddings[-1].permute(2, 0, 1)
            multi_stage_features[-1] = multi_stage_features[-1].permute(2, 0, 1)

        _, batch_size, _ = multi_stage_features[0].shape

        # [num_queries, batch_size, num_channels]
        query_embeddings = tm.queries_embedder.weight.unsqueeze(1).repeat(1, batch_size, 1)
        # steered_query_feat: (B, 100, 256) -> permute to (100, B, 256)
        query_features = steered_query_feat.permute(1, 0, 2)

        decoder_output = tm.decoder(
            inputs_embeds=query_features,
            multi_stage_positional_embeddings=multi_stage_positional_embeddings,
            pixel_embeddings=mask_features,
            encoder_hidden_states=multi_stage_features,
            query_position_embeddings=query_embeddings,
            feature_size_list=size_list,
            output_hidden_states=False,
            output_attentions=False,
            return_dict=True,
        )
        return decoder_output

    def forward(self, pixel_values, mask_labels=None, class_labels=None):
        """
        Forward pass with dynamic query steering.
        Args:
            pixel_values (Tensor): (B, 3, H, W)
            mask_labels (list of Tensors, optional): GT binary masks per image for training
            class_labels (list of Tensors, optional): GT class IDs per image for training
        Returns:
            dict containing:
                - 'loss': total loss (if mask_labels is provided)
                - 'masks_queries_logits': (B, 100, H/4, W/4)
                - 'class_queries_logits': (B, 100, num_classes + 1)
                - 'pred_junction_coords': (B, 4, 2) in [0, 1]^2
                - 'pred_junction_vis': (B, 4) visibility logits
                - 'junction_attn_weights': (B, 100, 4)
                - 'm2f_loss': standard Hungarian loss (if training)
        """
        B = pixel_values.shape[0]
        
        # 1. Swin-Tiny Backbone + MSDeformAttn Pixel Decoder
        pixel_level_outputs = self.m2f.model.pixel_level_module(pixel_values, output_hidden_states=True)
        # multi_scale_features has 3 tensors: [stride 32, stride 16, stride 8]
        multi_scale_features = list(pixel_level_outputs.decoder_hidden_states)
        mask_features = pixel_level_outputs.decoder_last_hidden_state # (B, 256, H/4, W/4)
        
        # 2. Extract 4 Anatomical Junctions from stride-16 features
        stride_16_feat = multi_scale_features[1] # (B, 256, H_16, W_16)
        pred_j_coords, pred_j_vis, j_features = self.junction_head(stride_16_feat)
        
        # 3. Dynamic Query Steering:
        # Base query features from Mask2Former
        tm = self.m2f.model.transformer_module
        base_query_feat = tm.queries_features.weight.unsqueeze(0).repeat(B, 1, 1) # (B, 100, 256)
        
        # Steer the queries with the 4 junction vectors
        steered_query_feat, j_attn_weights = self.query_steering(base_query_feat, j_features, pred_j_coords)
        
        # 4. Forward through Mask2Former Transformer Decoder with steered queries
        decoder_output = self.forward_transformer_decoder(
            multi_scale_features,
            mask_features,
            steered_query_feat
        )
        
        # Extract class & mask logits
        class_queries_logits = self.m2f.class_predictor(decoder_output.last_hidden_state) # (B, 100, num_classes + 1)
        masks_queries_logits = decoder_output.masks_queries_logits[-1]                    # (B, 100, H/4, W/4)
        
        output = {
            'masks_queries_logits': masks_queries_logits,
            'class_queries_logits': class_queries_logits,
            'pred_junction_coords': pred_j_coords,
            'pred_junction_vis': pred_j_vis,
            'junction_attn_weights': j_attn_weights
        }
        
        # 5. Compute Hungarian Matching Loss if GT labels are provided
        if mask_labels is not None and class_labels is not None:
            # Build loss using Mask2Former's built-in criterion
            loss_dict = self.m2f.criterion(
                masks_queries_logits=masks_queries_logits,
                class_queries_logits=class_queries_logits,
                mask_labels=mask_labels,
                class_labels=class_labels
            )
            # Weighted sum according to HuggingFace weight_dict
            weight_dict = self.m2f.criterion.weight_dict
            m2f_loss = sum(loss_dict[k] * weight_dict[k] for k in loss_dict.keys() if k in weight_dict)
            output['m2f_loss'] = m2f_loss
            output['m2f_loss_dict'] = loss_dict
            
        return output

def load_junction_steered_model(checkpoint_path=None, model_name="facebook/mask2former-swin-tiny-ade-semantic", device="cpu"):
    """
    Helper to instantiate and load checkpoint weights.
    """
    model = JunctionSteeredMask2Former(model_name=model_name, num_labels=4)
    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
        model.load_state_dict(state_dict)
        print(f"✅ Loaded checkpoint from: {checkpoint_path}")
    return model.to(device)

In [ ]:
# ==============================================================================
# 6. Multi-Task Loss Function
# ==============================================================================
"""
Multi-Task Loss for Junction-Steered Mask2Former (EXPERIMENT_5).
Combines:
  1. Mask2Former Hungarian Matching Loss (Classification Focal + Mask BCE + Mask Dice)
  2. Visibility-Masked Coordinate Smooth-L1 Loss on 4 Biological Junctions
  3. Binary Cross-Entropy on Junction Visibility Flags
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class JunctionSteeredLoss(nn.Module):
    def __init__(self, lambda_m2f=1.0, lambda_coord=5.0, lambda_vis=1.0):
        super().__init__()
        self.lambda_m2f = lambda_m2f
        self.lambda_coord = lambda_coord
        self.lambda_vis = lambda_vis
        self.bce_loss = nn.BCEWithLogitsLoss()

    def forward(self, model_outputs, gt_junction_coords, gt_junction_vis):
        """
        Args:
            model_outputs (dict): Output dict from JunctionSteeredMask2Former containing:
                - 'm2f_loss': scalar Hungarian loss
                - 'pred_junction_coords': (B, 4, 2) in [0, 1]^2
                - 'pred_junction_vis': (B, 4) visibility logits
            gt_junction_coords (Tensor): (B, 4, 2) normalized GT coordinates
            gt_junction_vis (Tensor): (B, 4) binary GT visibility flags {0, 1}
        Returns:
            total_loss (Tensor): scalar
            loss_dict (dict): component breakdowns
        """
        device = gt_junction_coords.device
        pred_coords = model_outputs['pred_junction_coords'] # (B, 4, 2)
        pred_vis_logits = model_outputs['pred_junction_vis'] # (B, 4)
        
        # 1. Mask2Former Base Loss
        m2f_loss = model_outputs.get('m2f_loss', torch.tensor(0.0, device=device))
        
        # 2. Junction Visibility Loss (BCE)
        vis_loss = self.bce_loss(pred_vis_logits, gt_junction_vis)
        
        # 3. Visibility-Masked Coordinate Loss (Smooth L1)
        # Coordinate loss is only computed for junctions visible in GT
        vis_mask = (gt_junction_vis > 0.5) # (B, 4)
        if vis_mask.sum() > 0:
            # Expand mask for 2D coords: (B, 4, 2)
            coord_mask = vis_mask.unsqueeze(-1).expand_as(pred_coords)
            diff = F.smooth_l1_loss(
                pred_coords[coord_mask],
                gt_junction_coords[coord_mask],
                beta=0.02,
                reduction='mean'
            )
            coord_loss = diff
        else:
            coord_loss = torch.tensor(0.0, device=device)
            
        # Total Weighted Multi-Task Loss
        total_loss = (self.lambda_m2f * m2f_loss +
                      self.lambda_coord * coord_loss +
                      self.lambda_vis * vis_loss)
                      
        loss_dict = {
            'loss': total_loss.item(),
            'm2f_loss': m2f_loss.item() if isinstance(m2f_loss, torch.Tensor) else m2f_loss,
            'coord_loss': coord_loss.item() if isinstance(coord_loss, torch.Tensor) else coord_loss,
            'vis_loss': vis_loss.item() if isinstance(vis_loss, torch.Tensor) else vis_loss
        }
        
        return total_loss, loss_dict

In [ ]:
# ==============================================================================
# 7. Post-Processing & Evaluation Metrics
# ==============================================================================
COLOR_MAP_RGB = {
    0: (0, 0, 0),       # BG
    1: (34, 197, 94),   # Ridge: Green
    2: (239, 68, 68),   # Silhouette: Red
    3: (59, 130, 246)   # Falciform: Blue
}

JUNCTION_COLORS_BGR = {
    0: (0, 215, 255), # Top: Gold
    1: (255, 255, 0), # Bottom: Cyan
    2: (255, 0, 255), # Lat_Right: Magenta
    3: (0, 140, 255)  # Lat_Left: Orange
}

def rasterize_class_map(masks_queries_logits, class_queries_logits, canvas_size=1024):
    """
    Standard Mask2Former post-processing: Argmax over class probabilities * sigmoid mask probabilities.
    Matches EXPERIMENT_5 implementation.
    """
    masks_queries_logits = masks_queries_logits.float()
    class_queries_logits = class_queries_logits.float()
    
    # 1. Resize mask logits to canvas_size
    masks = torch.nn.functional.interpolate(
        masks_queries_logits.unsqueeze(0),
        size=(canvas_size, canvas_size),
        mode='bilinear',
        align_corners=False
    ).squeeze(0).sigmoid() # (100, H, W)
    
    # 2. Query classification probabilities
    cls_probs = torch.softmax(class_queries_logits, dim=-1) # (100, 5), last is BG
    
    # Exclude background class (index 4)
    fg_cls_probs = cls_probs[:, :4] # (100, 4)
    
    # Multiply: sem_probs = sum_q (cls_prob * mask_prob)
    sem_probs = torch.einsum('qc,qhw->chw', fg_cls_probs, masks) # (4, H, W)
    
    # Thresholding & argmax in NumPy
    pred_map = sem_probs.argmax(dim=0).cpu().numpy().astype(np.int64) # (H, W)
    max_prob = sem_probs.max(dim=0)[0].cpu().numpy()
    pred_map[max_prob < 0.25] = 0
    
    return pred_map

def render_frame_diagnostic(orig_rgb_norm, gt_mask, pred_map, pred_j_coords, gt_j_coords, gt_j_vis, out_path, metrics):
    """
    Renders standard 4-panel diagnostic montage (Input, GT, Pred, Error Map).
    """
    img = (orig_rgb_norm.transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN) * 255.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    H, W = bgr.shape[:2]
    
    # Panel 2: GT Overlay
    p2 = bgr.copy()
    for c_id in [1, 2, 3]:
        mask_c = (gt_mask == c_id)
        if mask_c.any():
            col = COLOR_MAP_RGB[c_id][::-1]
            p2[mask_c] = (p2[mask_c] * 0.4 + np.array(col) * 0.6).astype(np.uint8)
    for k in range(4):
        if gt_j_vis[k] > 0.5:
            pt = (gt_j_coords[k] * float(W)).astype(int)
            cv2.circle(p2, tuple(pt), 14, (0, 0, 0), -1)
            cv2.circle(p2, tuple(pt), 10, JUNCTION_COLORS_BGR[k], -1)
            cv2.circle(p2, tuple(pt), 3, (255, 255, 255), -1)
            
    # Panel 3: Pred Overlay
    p3 = bgr.copy()
    for c_id in [1, 2, 3]:
        mask_c = (pred_map == c_id)
        if mask_c.any():
            col = COLOR_MAP_RGB[c_id][::-1]
            p3[mask_c] = (p3[mask_c] * 0.4 + np.array(col) * 0.6).astype(np.uint8)
    for k in range(4):
        pt = (pred_j_coords[k] * float(W)).astype(int)
        cv2.circle(p3, tuple(pt), 14, (0, 0, 0), -1)
        cv2.circle(p3, tuple(pt), 10, JUNCTION_COLORS_BGR[k], -1)
        cv2.circle(p3, tuple(pt), 3, (0, 0, 0), -1)
        
    # Panel 4: Error Map (TP=Green, FP=Cyan, FN=Red)
    p4 = bgr.copy()
    tp = (pred_map > 0) & (gt_mask > 0)
    fp = (pred_map > 0) & (gt_mask == 0)
    fn = (pred_map == 0) & (gt_mask > 0)
    
    p4[tp] = (p4[tp] * 0.3 + np.array([0, 255, 0]) * 0.7).astype(np.uint8)
    p4[fp] = (p4[fp] * 0.3 + np.array([255, 255, 0]) * 0.7).astype(np.uint8)
    p4[fn] = (p4[fn] * 0.3 + np.array([0, 0, 255]) * 0.7).astype(np.uint8)
    
    top_row = np.hstack([bgr, p2])
    bot_row = np.hstack([p3, p4])
    montage = np.vstack([top_row, bot_row])
    
    dice_val = metrics.get('macro_dice', 0.0)
    assd_val = metrics.get('macro_assd', 0.0)
    cv2.putText(montage, '1. Input Laparoscopic Frame', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(montage, '2. Ground Truth + GT Junctions', (W + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    cv2.putText(montage, f'3. Prediction (Dice: {dice_val:.1%}, ASSD: {assd_val:.1f}px)', (20, H + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 200), 2)
    cv2.putText(montage, '4. Error Map: Green=TP, Cyan=FP, Red=FN', (W + 20, H + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
    
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, montage, [cv2.IMWRITE_JPEG_QUALITY, 88])

def run_evaluation(model, dataloader, device, split_name='Val', out_dir=None, save_diagnostics=False):
    model.eval()
    records = []
    latencies = []
    
    diag_dir = os.path.join(out_dir, f'diagnostics_{split_name.lower()}') if (out_dir and save_diagnostics) else None
    print(f'\n🔍 Running {split_name} Evaluation ({len(dataloader.dataset)} frames)...')
    
    with torch.no_grad():
        for idx, batch in enumerate(dataloader):
            pixel_values = batch['pixel_values'].to(device)
            masks = batch['mask'].cpu().numpy()
            gt_j_coords = batch['junction_coords'].cpu().numpy()
            gt_j_vis = batch['junction_vis'].cpu().numpy()
            filenames = batch['filename']
            
            t0 = time.time()
            outputs = model(pixel_values)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            latencies.append((time.time() - t0) * 1000.0 / float(pixel_values.shape[0]))
            
            pred_masks_logits = outputs['masks_queries_logits']
            pred_cls_logits = outputs['class_queries_logits']
            pred_j_coords_t = outputs['pred_junction_coords'].float().cpu().numpy()
            
            for b in range(pixel_values.shape[0]):
                pred_map = rasterize_class_map(pred_masks_logits[b], pred_cls_logits[b])
                
                m = evaluate_frame_metrics(
                    pred_map=pred_map,
                    target_map=masks[b],
                    pred_j_coords=pred_j_coords_t[b],
                    gt_j_coords=gt_j_coords[b],
                    gt_j_vis=gt_j_vis[b]
                )
                m['filename'] = filenames[b]
                parts = filenames[b].split('_')
                pat_id = f'{parts[0]}_{parts[1]}' if len(parts) >= 2 else 'Unknown'
                m['patient'] = pat_id
                records.append(m)
                
                if diag_dir:
                    stem = Path(filenames[b]).stem
                    out_diag_path = os.path.join(diag_dir, f'{stem}_diag.jpg')
                    render_frame_diagnostic(
                        orig_rgb_norm=pixel_values[b].float().cpu().numpy(),
                        gt_mask=masks[b],
                        pred_map=pred_map,
                        pred_j_coords=pred_j_coords_t[b],
                        gt_j_coords=gt_j_coords[b],
                        gt_j_vis=gt_j_vis[b],
                        out_path=out_diag_path,
                        metrics=m
                    )
                    
            if (idx + 1) % 20 == 0 or (idx + 1) == len(dataloader):
                cur_dice = np.mean([r['macro_dice'] for r in records])
                cur_assd = np.mean([r['macro_assd'] for r in records])
                print(f'   [{split_name} {idx+1}/{len(dataloader)}] Macro Dice: {cur_dice:.2%} | Macro ASSD: {cur_assd:.2f}px')

    df = pd.DataFrame(records)
    summary = {
        'split': split_name,
        'total_frames': len(df),
        'macro_dice': float(df['macro_dice'].mean()),
        'macro_iou': float(df['macro_iou'].mean()),
        'macro_assd': float(df['macro_assd'].mean()),
        'ridge_dice': float(df['ridge_dice'].mean()),
        'sil_dice': float(df['sil_dice'].mean()),
        'falc_dice': float(df['falc_dice'].mean()),
        'fg_dice': float(df['fg_dice'].mean()),
        'mean_latency_ms': float(np.mean(latencies)),
        'fps': float(1000.0 / np.mean(latencies)),
        'gpu_name': torch.cuda.get_device_name(device) if device.type == 'cuda' else 'CPU/MPS'
    }
    
    pat_stats = {}
    for pat, pat_group in df.groupby('patient'):
        pat_stats[pat] = {
            'frames': len(pat_group),
            'macro_dice': float(pat_group['macro_dice'].mean()),
            'macro_iou': float(pat_group['macro_iou'].mean()),
            'macro_assd': float(pat_group['macro_assd'].mean()),
            'ridge_dice': float(pat_group['ridge_dice'].mean()),
            'sil_dice': float(pat_group['sil_dice'].mean()),
            'falc_dice': float(pat_group['falc_dice'].mean())
        }
    summary['per_patient_metrics'] = pat_stats
    
    valid_j = df['mean_j_err_px'].dropna()
    if len(valid_j) > 0:
        summary['mean_junction_err_px'] = float(valid_j.mean())
        
    return summary, df


In [ ]:
# ==============================================================================
# 8. Main Training Execution
# ==============================================================================
def collate_fn_l3d(batch):
    pixel_values = torch.stack([b['pixel_values'] for b in batch])
    masks = torch.stack([b['mask'] for b in batch])
    junction_coords = torch.stack([b['junction_coords'] for b in batch])
    junction_vis = torch.stack([b['junction_vis'] for b in batch])
    filenames = [b['filename'] for b in batch]
    is_p40s = [b['is_patient_40'] for b in batch]
    
    mask_labels_list = []
    class_labels_list = []
    
    for b in range(len(batch)):
        m = masks[b]
        unique_classes = torch.unique(m)
        unique_classes = unique_classes[unique_classes > 0]
        
        if len(unique_classes) == 0:
            mask_labels_list.append(torch.zeros((1, m.shape[0], m.shape[1]), dtype=torch.float32))
            class_labels_list.append(torch.zeros(1, dtype=torch.int64))
        else:
            binary_masks = torch.stack([(m == c).float() for c in unique_classes])
            mask_labels_list.append(binary_masks)
            class_labels_list.append(unique_classes.long())
            
    return {
        'pixel_values': pixel_values,
        'mask': masks,
        'mask_labels': mask_labels_list,
        'class_labels': class_labels_list,
        'junction_coords': junction_coords,
        'junction_vis': junction_vis,
        'filename': filenames,
        'is_patient_40': is_p40s
    }

def build_optimizer_and_scheduler(model, lr_backbone=1e-5, lr_head=1e-4, weight_decay=1e-4, epochs=60):
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'pixel_level_module.encoder' in name:
            backbone_params.append(param)
        else:
            head_params.append(param)
            
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone},
        {'params': head_params, 'lr': lr_head}
    ], weight_decay=weight_decay)
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    return optimizer, scheduler

EPOCHS = 60
BATCH_SIZE = 2
ACCUM_STEPS = 2
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5

train_dataset = StratifiedL3DDataset(split='Train', train_dir=TRAIN_DIR, val_dir=VAL_DIR)
val_dataset = StratifiedL3DDataset(split='Val', train_dir=TRAIN_DIR, val_dir=VAL_DIR)
test_dataset = StratifiedL3DDataset(split='Test', test_dir=TEST_DIR)

num_workers = 2 if device.type == 'cuda' else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, collate_fn=collate_fn_l3d)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, collate_fn=collate_fn_l3d)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, collate_fn=collate_fn_l3d)

model = JunctionSteeredMask2Former(num_labels=4).to(device)
criterion = JunctionSteeredLoss(lambda_m2f=1.0, lambda_coord=5.0, lambda_vis=1.0)
optimizer, scheduler = build_optimizer_and_scheduler(model, lr_backbone=LR_BACKBONE, lr_head=LR_HEAD, epochs=EPOCHS)

use_amp = (device.type == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f"🚀 Starting {EPOCHS}-epoch training on {device}...")
best_val_dice = -1.0
best_ckpt_path = os.path.join(OUT_DIR, 'best_model.pth')
training_log = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    losses = []
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        mask_labels = [m.to(device) for m in batch['mask_labels']]
        class_labels = [c.to(device) for c in batch['class_labels']]
        gt_j_coords = batch['junction_coords'].to(device)
        gt_j_vis = batch['junction_vis'].to(device)
        
        with torch.amp.autocast('cuda', enabled=use_amp, dtype=amp_dtype):
            outputs = model(pixel_values, mask_labels=mask_labels, class_labels=class_labels)
            total_loss, loss_dict = criterion(outputs, gt_j_coords, gt_j_vis)
            loss_scaled = total_loss / float(ACCUM_STEPS)
            
        scaler.scale(loss_scaled).backward()
        losses.append(loss_dict['loss'])
        
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
    scheduler.step()
    
    # Validation evaluation
    val_summary, val_df = run_evaluation(model, val_loader, device, split_name='Val', out_dir=OUT_DIR)
    vdice = val_summary['macro_dice']
    vassd = val_summary['macro_assd']
    
    log_entry = {
        'epoch': epoch,
        'train_loss': float(np.mean(losses)),
        'val_macro_dice': vdice,
        'val_macro_assd': vassd,
        'duration_sec': time.time() - t0
    }
    training_log.append(log_entry)
    pd.DataFrame(training_log).to_csv(os.path.join(OUT_DIR, 'training_log.csv'), index=False)
    
    print(f"Epoch [{epoch}/{EPOCHS}] Train Loss: {np.mean(losses):.4f} | Val Dice: {vdice:.2%} | ASSD: {vassd:.2f}px ({time.time()-t0:.1f}s)")
    
    if vdice > best_val_dice:
        best_val_dice = vdice
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_macro_dice': vdice}, best_ckpt_path)
        print(f"⭐ New best model saved: {best_val_dice:.2%}")

# Final Comprehensive Evaluation
print('\n🏁 Running Final Comprehensive Evaluation on Best Checkpoint...')
ckpt = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

val_summary, val_df = run_evaluation(model, val_loader, device, split_name='Val-Final', out_dir=OUT_DIR, save_diagnostics=True)
val_df.to_csv(os.path.join(OUT_DIR, 'val_predictions.csv'), index=False)

test_summary, test_df = run_evaluation(model, test_loader, device, split_name='Test-Final', out_dir=OUT_DIR, save_diagnostics=True)
test_df.to_csv(os.path.join(OUT_DIR, 'test_predictions.csv'), index=False)

full_metrics = {'val_summary': val_summary, 'test_summary': test_summary, 'best_epoch': ckpt.get('epoch', None)}
with open(os.path.join(OUT_DIR, 'metrics_summary.json'), 'w') as f:
    json.dump(full_metrics, f, indent=4)

# Zip results for download
zip_path = os.path.join(OUT_DIR, 'results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(OUT_DIR):
        for f in files:
            if f != 'results.zip':
                p = os.path.join(root, f)
                zipf.write(p, os.path.relpath(p, OUT_DIR))

print(f"🎉 All done! Download results archive from: {zip_path}")
